#  Модуль 5. Очистка данных (Data Cleaning)

## Подробный конспект

### 5.1. Зачем нужна очистка данных

Данные из реального мира редко бывают идеальными. Они приходят с пропусками, опечатками, дубликатами и противоречиями. Если скормить такие данные модели — она выучит мусор.

**Аналогия:** вы готовите ужин. Если в холодильнике есть испорченные продукты, вы не начнёте готовить, пока их не выбросите. Очистка данных — это именно этот этап: выбросить испорченное, отмыть овощи, разобрать, что к чему.

**Что мы чистим:**
- Пропущенные значения (пустые ячейки)
- Дубликаты (одна и та же запись дважды)
- Некорректные значения (возраст 250 лет, отрицательная цена)
- Несогласованность форматов (даты в разных форматах, разные единицы измерения)

### 5.2. Пропущенные значения (Missing Values)

#### Что такое пропуск в pandas

В Python пропуск может выглядеть по-разному, но pandas использует специальные объекты:

| Обозначение | Что это | Где встречается |
|-------------|---------|----------------|
| `NaN` | Not a Number | Числовые столбцы (float) |
| `None` | Пустой объект | Текстовые столбцы (object) |
| `NaT` | Not a Time | Столбцы с датами (datetime) |

> **Важно:** `NaN` — это **не текст** `"NaN"` и **не ноль** `0`. Это специальное значение, которое pandas понимает как «данных нет».

#### Причины появления пропусков

1. **Ошибки сбора данных**
   - Сенсор не сработал, форма не заполнена, отказ оборудования.
   - Пример: в анкете поле «доход» необязательное — многие пропускают.

2. **Ошибки слияния (merge/join)**
   - Вы соединили две таблицы, но не для всех записей нашлось соответствие.
   - Пример: таблица заказов + таблица клиентов. У нового клиента ещё нет профиля — поля из профиля пустые.

3. **Человеческий фактор**
   - Оператор ввёл данные не во все поля.
   - Пример: в базе клиентов 30% записей без телефона — менеджер забыл спросить.

4. **Особенность данных**
   - Просто так устроена реальность.
   - Пример: не все товары имеют скидку, не все пациенты сдают анализ крови.

#### Типы пропусков: MCAR, MAR, MNAR

Эта классификация важна, потому что от неё зависит, можно ли просто удалить пропуски или нужно более хитрое заполнение.

**MCAR — Missing Completely At Random**
*(Полностью случайные пропуски)*

Пропуски не зависят ни от каких данных — чистая случайность.

*Пример:* анкета разлетелась по ветру, несколько листов потерялись. Пропуски равномерно разбросаны по всему датасету.

**Что делать:** можно удалять строки или заполнять средним — искажений будет минимум.

**MAR — Missing At Random**
*(Случайные пропуски, зависящие от других признаков)*

Пропуск в одном столбце зависит от значений в других столбцах, но **не** от самого пропущенного значения.

*Пример:* мужчины реже указывают возраст в анкете, чем женщины. Пропуск в «возрасте» зависит от «пола», но не от самого возраста.

**Что делать:** заполнять средним **внутри группы** (средний возраст мужчин — для мужчин, женщин — для женщин).

**MNAR — Missing Not At Random**
*(Неслучайные пропуски)*

Пропуск зависит от самого пропущенного значения.

*Пример:* люди с очень высоким доходом не хотят его указывать. Или пациенты с тяжёлой формой болезни не приходят на обследование.

**Что делать:** простое заполнение средним **исказит** картину. Нужно моделировать пропуски или использовать специальные методы.

#### Диагностика пропусков

**Базовые методы pandas:**

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'имя': ['Анна', 'Борис', 'Виктор', 'Галина'],
    'возраст': [25, np.nan, 35, np.nan],
    'доход': [50000, 60000, np.nan, 45000],
    'город': ['Москва', 'СПб', 'Москва', 'СПб']
})

# Проверка на пропуски (булева маска)
print(df.isnull())
#      имя  возраст  доход   город
# 0  False    False  False   False
# 1  False     True  False   False
# 2  False    False   True   False
# 3  False     True  False   False

# Количество пропусков по столбцам
print(df.isnull().sum())
# имя        0
# возраст    2
# доход      1
# город      0

# Доля пропусков в процентах
print((df.isnull().sum() / len(df) * 100).round(2))
# имя        0.0
# возраст   50.0
# доход     25.0

# Строки, где есть хотя бы один пропуск
print(df[df.isnull().any(axis=1)])

# Строки, где пропусков нет вообще
print(df.dropna())  # или df[df.notna().all(axis=1)]

**Визуализация пропусков через `missingno`:**

In [ ]:
import missingno as msno
import matplotlib.pyplot as plt

# Бар-чарт: сколько непустых значений в каждом столбце
msno.bar(df)
plt.show()

# Матрица пропусков: видно паттерны
# Если полосы пропусков совпадают — возможно, MAR (системные пропуски)
msno.matrix(df)
plt.show()

# Тепловая карта корреляции пропусков
# Показывает, появляются ли пропуски в двух столбцах вместе
msno.heatmap(df)
plt.show()

**Анализ паттернов:** если пропуски в «доходе» часто совпадают с пропусками в «образовании» — возможно, это MAR (одна и та же группа людей не заполняла раздел анкеты).

### 5.3. Стратегии обработки пропусков

#### Стратегия 1: Удаление (Deletion)

Самый простой способ, но самый опасный: вы теряете данные.

In [ ]:
# Удалить строки, где есть хотя бы один пропуск
df_clean = df.dropna()

# Удалить только если пропущен доход (конкретный столбец)
df_clean = df.dropna(subset=['доход'])

# Удалить столбцы, где пропусков больше 50%
threshold = 0.5
df_clean = df.loc[:, df.isnull().mean() < threshold]

# Удалить столбец, если он почти весь пустой
df_clean = df.drop(columns=['столбец_с_90%_пропусков'])

**Когда удалять:**
- Пропусков мало (< 5% от всех данных)
- Пропуски типа MCAR (случайные)
- Строка дублируется или не нужна

**Когда НЕ удалять:**
- Пропусков много (> 30%) — потеряете слишком много данных
- Пропуски типа MNAR — удаление исказит выборку

#### Стратегия 2: Заполнение константами

In [ ]:
# Заполнить все пропуски нулём
df['доход'] = df['доход'].fillna(0)

# Заполнить текстом "Unknown"
df['город'] = df['город'].fillna('Unknown')

# Заполнить специальным маркером (для MNAR)
df['доход'] = df['доход'].fillna(-999)  # модель поймёт, что это "особое" значение

**Когда использовать:**
- Ноль имеет смысл (например, количество детей — 0)
- Категории — «Unknown» лучше, чем удаление
- Создание индикатора пропуска (см. ниже)

#### Стратегия 3: Заполнение статистиками

In [ ]:
# Среднее (для симметричных распределений)
mean_age = df['возраст'].mean()
df['возраст'] = df['возраст'].fillna(mean_age)

# Медиана (для скошенных распределений и при наличии выбросов)
median_income = df['доход'].median()
df['доход'] = df['доход'].fillna(median_income)

# Мода (для категориальных признаков)
mode_city = df['город'].mode()[0]  # mode() возвращает Series
df['город'] = df['город'].fillna(mode_city)

**Правило выбора статистики:**
- **Среднее** — если распределение нормальное, без выбросов
- **Медиана** — если распределение скошено или есть выбросы
- **Мода** — для категориальных признаков

#### Стратегия 4: Заполнение по группам (Group-based Imputation)

Если пропуски MAR — заполняем статистикой внутри группы.

In [ ]:
# Заполняем пропущенный доход медианным доходом В ЭТОМ ГОРОДЕ
df['доход'] = df.groupby('город')['доход'].transform(
    lambda x: x.fillna(x.median())
)

# То же самое, но явно:
for city in df['город'].unique():
    mask = (df['город'] == city) & (df['доход'].isnull())
    median_city = df.loc[df['город'] == city, 'доход'].median()
    df.loc[mask, 'доход'] = median_city

**Почему это лучше среднего:** медианный доход в Москве и в деревне различается в разы. Заполнение общей медианой испортит обе группы.

#### Стратегия 5: Заполнение соседними значениями

Полезно для временных рядов.

In [ ]:
# Заполнить предыдущим значением (forward fill)
df['температура'] = df['температура'].fillna(method='ffill')
# или современный синтаксис:
df['температура'] = df['температура'].ffill()

# Заполнить следующим значением (backward fill)
df['температура'] = df['температура'].bfill()

# Интерполяция (линейная)
df['температура'] = df['температура'].interpolate(method='linear')

#### Стратегия 6: Продвинутые методы (ML-based Imputation)

**KNNImputer** — заполняет пропуск на основе K ближайших соседей по другим признакам.

In [ ]:
from sklearn.impute import KNNImputer

# Выбираем числовые признаки для поиска соседей
num_features = ['возраст', 'доход', 'стаж']
df_num = df[num_features].copy()

# K=5 ближайших соседей
imputer = KNNImputer(n_neighbors=5)
df_imputed = pd.DataFrame(
    imputer.fit_transform(df_num),
    columns=num_features,
    index=df.index
)

# Заменяем только пропущенные значения (чтобы не трогать существующие)
for col in num_features:
    df[col] = df[col].fillna(df_imputed[col])

**IterativeImputer (MICE)** — строит модель для предсказания пропущенных значений по остальным признакам, итеративно, пока не сойдётся.

In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

imputer = IterativeImputer(max_iter=10, random_state=42)
df_imputed = pd.DataFrame(
    imputer.fit_transform(df_num),
    columns=num_features,
    index=df.index
)

> **Важно:** продвинутые импутеры работают только с числовыми данными. Категории нужно предварительно закодировать.

#### Стратегия 7: Индикатор пропуска (Missing Indicator)

Иногда сам факт пропуска — это информация.

In [ ]:
# Создаём новый признак: был ли пропуск в доходе
df['доход_пропущен'] = df['доход'].isnull().astype(int)

# Затем заполняем доход медианой
df['доход'] = df['доход'].fillna(df['доход'].median())

**Зачем:** если MNAR (люди с высоким доходом скрывают его), модель увидит флаг `доход_пропущен=1` и поймёт, что это особая категория.

### 5.4. Дублированные записи

Дубликаты — строки, которые встречаются дважды или более. Они искажают статистику и заставляют модель переобучаться на повторяющихся примерах.

In [ ]:
# Проверить, есть ли дубликаты
print(df.duplicated().sum())  # количество полных дубликатов

# Посмотреть сами дубли
print(df[df.duplicated(keep=False)])  # keep=False показывает ВСЕ копии

# Удалить полные дубликаты (оставляем первое вхождение)
df = df.drop_duplicates()

# Удалить дубликаты по конкретным столбцам
# (если один и тот же клиент записан дважды с разным id)
df = df.drop_duplicates(subset=['имя', 'дата_рождения', 'город'], keep='first')

# keep='last' — оставить последнее вхождение
# keep=False — удалить ВСЕ дубли (даже оригинал)

**«Мягкие» дубликаты:** строки не идентичны полностью, но очень похожи (опечатки, разница в регистре). С ними сложнее — нужна нечёткая дедупликация (fuzzy matching), но это выходит за рамки базового курса.

### 5.5. Некорректные значения

#### Опечатки и вариации в категориях

In [ ]:
# Проблема: один и тот же город записан по-разному
df['город'].unique()
# ['Москва', 'москва', 'МСК', 'СПб', 'Санкт-Петербург', 'Piter']

# Решение: приведение к единому виду
df['город'] = df['город'].str.strip()           # убрать пробелы по краям
df['город'] = df['город'].str.lower()           # в нижний регистр

# Затем — ручная или автоматическая замена (mapping)
city_mapping = {
    'москва': 'Москва',
    'мск': 'Москва',
    'санкт-петербург': 'Санкт-Петербург',
    'piter': 'Санкт-Петербург',
    'спб': 'Санкт-Петербург'
}
df['город'] = df['город'].replace(city_mapping)

#### Неверные типы данных

In [ ]:
# Число записано как текст
df['возраст'] = pd.to_numeric(df['возраст'], errors='coerce')
# errors='coerce' превратит "неизвестно" в NaN

# Дата записана как текст
df['дата'] = pd.to_datetime(df['дата'], errors='coerce', dayfirst=True)

#### Нелогичные значения (Outliers по смыслу)

In [ ]:
# Возраст не может быть отрицательным или 250 лет
df.loc[df['возраст'] < 0, 'возраст'] = np.nan
df.loc[df['возраст'] > 120, 'возраст'] = np.nan

# Доход не может быть отрицательным
df.loc[df['доход'] < 0, 'доход'] = np.nan

# Процентная ставка не может быть 999% (часто маркер пропуска)
df.loc[df['ставка'] == 999, 'ставка'] = np.nan

### 5.6. Несогласованность форматов

#### Даты

In [ ]:
# Разные форматы в одном столбце
# '01.03.2024', '2024-03-01', '03/01/2024'

df['дата'] = pd.to_datetime(df['дата'], errors='coerce', dayfirst=True)

# Если форматы сильно разные — иногда проще через infer
df['дата'] = pd.to_datetime(df['дата'], infer_datetime_format=True)

#### Валюты и единицы измерения

In [ ]:
# Площадь: где-то в м², где-то в футах
# Решение: привести всё к м²
df['площадь_м2'] = df.apply(
    lambda row: row['площадь'] * 0.0929 if row['единица'] == 'ft' else row['площадь'],
    axis=1
)

# Валюта: разные столбцы или маркер валюты
# Лучше хранить валюту отдельно от суммы

#### Регистр, пробелы, спецсимволы в тексте

In [ ]:
df['имя'] = df['имя'].str.strip()                    # пробелы
df['имя'] = df['имя'].str.title()                    # 'иван' -> 'Иван'
df['телефон'] = df['телефон'].str.replace(r'\D', '', regex=True)  # оставить только цифры

### 5.7. Практический пример: полный цикл очистки

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
import missingno as msno

np.random.seed(42)

# Создаём «грязные» данные
df = pd.DataFrame({
    'id': range(1, 101),
    'имя': ['Анна', 'Борис', 'Виктор', 'Галина'] * 25,
    'возраст': np.concatenate([
        np.random.normal(35, 10, 90).clip(18, 70),
        [250, -5, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan]
    ]),
    'доход': np.concatenate([
        np.random.lognormal(11, 0.5, 85),
        [np.nan] * 15
    ]),
    'город': (['Москва', 'москва', 'МСК', 'СПб', 'СПб'] * 20),
    'дата_регистрации': pd.date_range('2023-01-01', periods=100, freq='D').astype(str).tolist()
})

# Добавим дубликат
df = pd.concat([df, df.iloc[[0]]], ignore_index=True)

print("ИСХОДНЫЕ ДАННЫЕ:")
print(f"Размер: {df.shape}")
print(f"Пропуски:\n{df.isnull().sum()}")
print(f"Дубликаты: {df.duplicated().sum()}")
print(f"Уникальные города: {df['город'].unique()}")

# === ШАГ 1: ДУБЛИКАТЫ ===
df = df.drop_duplicates()
print(f"\nПосле удаления дубликатов: {df.shape}")

# === ШАГ 2: ФОРМАТЫ И КАТЕГОРИИ ===
# Приведение города к единому виду
df['город'] = df['город'].str.strip().str.lower()
city_map = {'москва': 'Москва', 'мск': 'Москва', 'спб': 'Санкт-Петербург'}
df['город'] = df['город'].replace(city_map)

# Дата
df['дата_регистрации'] = pd.to_datetime(df['дата_регистрации'])

# === ШАГ 3: НЕЛОГИЧНЫЕ ЗНАЧЕНИЯ ===
df.loc[(df['возраст'] < 0) | (df['возраст'] > 120), 'возраст'] = np.nan

# === ШАГ 4: ПРОПУСКИ ===
# Создаём индикаторы
df['возраст_пропущен'] = df['возраст'].isnull().astype(int)
df['доход_пропущен'] = df['доход'].isnull().astype(int)

# Заполняем возраст медианой по городу
df['возраст'] = df.groupby('город')['возраст'].transform(lambda x: x.fillna(x.median()))

# Заполняем доход медианой по городу
df['доход'] = df.groupby('город')['доход'].transform(lambda x: x.fillna(x.median()))

# Если после группировки остались пропуски (город без данных) — общая медиана
df['возраст'] = df['возраст'].fillna(df['возраст'].median())
df['доход'] = df['доход'].fillna(df['доход'].median())

print("\nПОСЛЕ ОЧИСТКИ:")
print(f"Пропуски:\n{df.isnull().sum()}")
print(f"Уникальные города: {df['город'].unique()}")
print(df[['возраст', 'доход', 'город', 'возраст_пропущен', 'доход_пропущен']].head(10))

### 5.8. Чек-лист для самопроверки

Перед переходом к Модулю 6 убедитесь, что вы:

- [ ] Понимаете разницу между `NaN`, `None` и `NaT`
- [ ] Можете объяснить MCAR, MAR и MNAR на примерах
- [ ] Умеете находить пропуски через `.isnull()` и визуализировать через `missingno`
- [ ] Знаете, когда можно удалять строки с пропусками, а когда нельзя
- [ ] Можете заполнить пропуски средним, медианой и модой
- [ ] Умеете заполнять пропуски по группам (`.groupby().transform()`)
- [ ] Знаете, зачем создавать индикатор пропуска
- [ ] Можете удалять дубликаты полностью и по подмножеству столбцов
- [ ] Умеете находить и исправлять опечатки в категориальных данных
- [ ] Можете обрабатывать нелогичные значения (возраст 250, отрицательный доход)
- [ ] Умеете приводить даты к единому формату
- [ ] Понимаете, почему важно привести категории к единому регистру и убрать пробелы

> **Переход к Модулю 6:** Данные очищены от пропусков и дубликатов, но в них всё ещё могут быть выбросы — значения, которые сильно отличаются от основной массы. Следующий модуль научит находить аномалии статистически и с помощью ML-методов, а также правильно с ними поступать.